# Build IIA Dataset — `build_iia_dataset_20260323.py`

**Purpose:** Construct the bilateral investment treaty (BIT) analysis dataset from raw source data for Horn & Sanctuary (2026).

**Output:** `iia_dataset_for_paper_20260323.csv` — 1,837 rows × 192 columns, one row per in-force BIT with ISO-resolved party codes.

---

## Input files

| # | File | Description |
|---|------|-------------|
| 1 | `data/IIA_raw/iia_dict_26-02-12.json` | Scraped UNCTAD IIA mapping database (2,808 entries, nested JSON) |
| 2 | `data/rename_datacols.csv` | JSON path → short column-name mapping (107 rows) |
| 3 | `data/investment_raw/iso_2024-04-09.xlsx` | Country name → ISO 3166-1 alpha-3 lookup |
| 4 | `data/biodiversity_raw/.../full_species_list_dec2023_final_240108_renamed.csv` | IUCN Red List species assessments (26,198 species) |
| 5 | `data/biodiversity_raw/.../spp_pp_2023_final_240111_renamed.csv` | Species range fractions by country (171,655 rows) |
| 6 | `[OUTPUT_DIR]/oecd_fdi_outward_raw.csv` | OECD bilateral outward FDI positions (mean 2021–2024, USD millions) |

## Pipeline steps

1. Load and flatten the IIA JSON dictionary  
2. Convert to DataFrame and apply column renaming  
3. Add ISO3 codes for party1 / party2  
4. Standardise key variable values  
5. Filter to BITs in force with both ISO codes resolved  
6. Apply `isds_limitprovisions` special-case overrides (8 treaties)  
7. Compute the 14-criterion `protective` indicator  
8. Merge OECD bilateral FDI positions (mean 2021–2024)  
9. Compute `EXT_BD_c` from raw IUCN species data  
10. Compute `t_treaties` indicator  
11. Add country-group indicators (LDC / megadiverse / OECD)  
12. Rename columns and save  

---

## `EXT_BD_c` methodology

For each country $c$:

$$E_{\text{cost},c} = \sum_s w(\text{cat}_s) \times pp_{s,c}$$

where $pp_{s,c}$ is the fraction of species $s$'s global range in country $c$, and $w(\text{cat}_s)$ is the IUCN status weight:

| Status | Weight |
|--------|--------|
| LC | 0.0 |
| NT | 0.2 |
| VU | 0.4 |
| EN | 0.6 |
| CR / CR(PE) / CR(PEW) | 0.8 |
| EX / EW | 1.0 |
| DD | excluded |

Normalised so the most extinction-exposed country scores 1:

$$\text{EXT\_BD\_c}_c = \frac{E_{\text{cost},c}}{\max_c(E_{\text{cost},c})}$$


## Step 0 — Imports and configuration

Edit the path variables below to match your local repository layout before running.

In [1]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import os

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd


In [2]:
# ═════════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — self-contained replication package
# ══════════════════════════════════════════════════════════════════════════════════
# All inputs are read from DATA_DIR; all outputs are written to OUTPUT_DIR
# (= DATA_DIR/output). Point DATA_DIR at the replication folder and the
# notebook runs end-to-end without any other path edits.

# Root directories
# By default DATA_DIR is the folder this notebook lives in (the replication
# package), so the notebook runs with no path edits when Jupyter is launched
# from that folder. To run from elsewhere, set DATA_DIR to an absolute path.
DATA_DIR = os.getcwd()
OUTPUT_DIR = os.path.join(DATA_DIR, "output")

# ── Input files (all directly inside DATA_DIR) ──────────────────────────────
IIA_DICT_FILE = "iia_dict_26-02-12.json"
IIA_DICT_PATH = os.path.join(DATA_DIR, IIA_DICT_FILE)
RENAME_CSV    = os.path.join(DATA_DIR, "rename_datacols.csv")
ISO_XLSX      = os.path.join(DATA_DIR, "iso_2024-04-09.xlsx")
SPP_LIST_CSV  = os.path.join(DATA_DIR, "full_species_list_dec2023_final_240108_renamed.csv")
SPP_PP_CSV    = os.path.join(DATA_DIR, "spp_pp_2023_final_240111_renamed.csv")
OECD_FDI_CSV  = os.path.join(DATA_DIR, "oecd_fdi_outward_raw.csv")
TAX_HARM_CSV  = os.path.join(DATA_DIR, "tax_revenue_harmonised_2018_2024.csv")
TAX_WB_CSV    = os.path.join(DATA_DIR, "tax_revenue_2024_worldbank.csv")

# ── Output file ───────────────────────────────────────────────────
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "iia_dataset_for_paper_20260323.csv")

# ── T_treaties thresholds ───────────────────────────────────────────
FDI_THRESHOLD_mUSD = 1_000   # USD 1 billion expressed in millions
EXT_THRESHOLD      = 0.1

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuration set.")
print(f"  DATA_DIR   : {DATA_DIR}")
print(f"  OUTPUT_DIR : {OUTPUT_DIR}")
print(f"  OUTPUT_CSV : {OUTPUT_CSV}")


Configuration set.
  DATA_DIR   : /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data
  OUTPUT_DIR : /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data/output
  OUTPUT_CSV : /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data/output/iia_dataset_for_paper_20260323.csv


## Step 1 — Country-name normalisation

`country_name_change()` converts raw country name strings (from the UNCTAD JSON and ISO lookup table) to a consistent lowercase canonical form used throughout the pipeline. This ensures reliable name-to-ISO matching downstream.


In [3]:
def country_name_change(name: str) -> str:
    """Normalise a country name string to the project's canonical lowercase form."""
    name = str(name).lower().strip()

    if "united states of america" in name:       return "united states"
    if "united kingdom" in name:                 return "united kingdom"
    if "belg" in name:                           return "bleu belg"
    if "luxem" in name:                          return "bleu lux"
    if "russia" in name:                         return "russian federation"
    if "czechia" in name:                        return "czech republic"
    if "slovak" in name:                         return "slovakia"
    if "moldova" in name:                        return "republic of moldova"
    if "bosnia" in name:                         return "bosnia and herzegovina"
    if "congo, democratic republic of" in name \
            or "congo, dem. rep." in name \
            or "congo drc" in name:              return "democratic republic of the congo"
    if "congo, rep." in name:                    return "congo"
    if "tanzania" in name:                       return "united republic of tanzania"
    if "venezuela" in name:                      return "bolivarian republic of venezuela"
    if "bolivia" in name:                        return "plurinational state of bolivia"
    if "iran" in name:                           return "islamic republic of iran"
    if "kyrgyz" in name:                         return "kyrgyzstan"
    if "syrian arab republic" in name:           return "syria"
    if "pale" in name:                           return "palestine"
    if "yemen" in name:                          return "yemen"
    if "egypt" in name:                          return "egypt"
    if "gambia, the" in name:                    return "gambia"
    if "côte d'ivoire" in name:                  return "cote d'ivoire"
    if "türkiye" in name:                        return "turkiye"
    if "viet nam" in name:                       return "vietnam"
    if "lao people" in name or "lao pdr" in name: return "laos"
    if "china" in name:
        if "macao" in name:      return "china - macao"
        if "hong kong" in name:  return "china - hong kong"
        if "taiwan" in name:     return "china - taiwan"
        return "china"
    if "macao" in name:                          return "china - macao"
    if "hong kong" in name:                      return "china - hong kong"
    if "taiwan" in name:                         return "taiwan - hong kong"
    if "korea" in name:
        return "north korea" if ("dem" in name or "north" in name) else "south korea"
    if "falkland" in name:                       return "falkland islands"
    if "bahamas" in name:                        return "bahamas"
    if "micronesia" in name:                     return "micronesia"
    if "lucia" in name:                          return "saint lucia"
    if "nevis" in name:                          return "saint kitts and nevis"
    if "martinique" in name:                     return "martinique"
    if "martin" in name:                         return "saint martin (french part)"
    if "vincent" in name:                        return "saint vincent and the grenadines"
    if "virgin" in name:
        return "us virgin islands" if "u" in name else "british virgin islands"
    if "bonaire" in name:                        return "bonaire"
    if "cocos" in name:                          return "cocos"
    if "maarten" in name:                        return "sint maarten"
    if "wallis" in name:                         return "wallis and futuna"
    if "solomon" in name:                        return "solomon islands"
    if "togo" in name:                           return "togo"
    return name

print("country_name_change() defined.")


country_name_change() defined.


## Step 2 — Flatten the IIA JSON dictionary

The raw UNCTAD data is a deeply nested JSON structure. `flatten_iia_dict()` recursively flattens each treaty's metadata into a single dict keyed by hierarchical path strings (e.g. `"FET Type - FET unqualified"`), ready for conversion to a rectangular DataFrame.


In [4]:
def _get_iia_metadata(iia_dict: dict, party1: str, party2: str) -> dict:
    """Recursively flatten per-IIA metadata into a single flat dict."""
    data_dict = {}
    for key, data in iia_dict.items():
        if isinstance(data, dict):
            if key == "related_agreements":
                data_dict[key] = [data[k] for k in data]
            else:
                for nk1, nd1 in data.items():
                    if isinstance(nd1, dict):
                        for nk2, nd2 in nd1.items():
                            if isinstance(nd2, dict):
                                for nk3, nd3 in nd2.items():
                                    data_dict[f"{key} - {nk1} -- {nk2} --- {nk3}"] = str(nd3)
                            else:
                                data_dict[f"{key} - {nk1} -- {nk2}"] = str(nd2)
                    else:
                        data_dict[f"{key} - {nk1}"] = str(nd1)
        else:
            if key == "Parties":
                data_dict["party1"] = party1
                data_dict["party2"] = party2
            else:
                if isinstance(data, list) and data:
                    data = data[0]
                elif not data:
                    data = np.nan
                data_dict[key] = data
    return data_dict


def flatten_iia_dict(iia_dict: dict) -> dict:
    """Iterate over every IIA and return a flat dict keyed by treaty filename."""
    flattened = {}
    iias = iia_dict.get("iias", {})
    for filename, iia in iias.items():
        meta   = iia.get("meta_data", {})
        part1  = meta.get("part1", {})
        part2  = meta.get("part2", {})
        parties = part1.get("Parties", [])
        if not isinstance(parties, list):
            continue
        party1 = country_name_change(parties[0].replace("1. ", ""))
        party2 = country_name_change(parties[1].replace("2. ", "") if len(parties) > 1 else "")
        part2_clean = {k: v for k, v in part2.items()
                       if k not in ("originally mapped", "mapped by")}
        merged   = {**part1, **part2_clean}
        flat_row = _get_iia_metadata(merged, party1, party2)
        flattened[filename] = flat_row
    return flattened

print("flatten_iia_dict() defined.")


flatten_iia_dict() defined.


## Step 3 — Convert to DataFrame and rename columns

Stack the per-treaty flat dicts into a pandas DataFrame, then apply the `rename_datacols.csv` mapping to replace long hierarchical path strings with short, readable column names (e.g. `"FET_Type"`, `"Exprop_Scope"`).


In [5]:
def dict_to_dataframe(flat_iia_dict: dict, rename_csv: str) -> pd.DataFrame:
    """Stack flat dicts into a DataFrame and apply column-name renaming."""
    clean_rows = []
    for row in flat_iia_dict.values():
        clean = {k: (v[0] if isinstance(v, list) and len(v) == 1
                     else (str(v) if isinstance(v, list) else v))
                 for k, v in row.items()}
        clean_rows.append(clean)
    iia_df    = pd.DataFrame(clean_rows)
    rename_df  = pd.read_csv(rename_csv)
    rename_map = dict(zip(rename_df["old_colnames"], rename_df["IIA_Datatype"]))
    iia_df     = iia_df.rename(columns=rename_map)
    return iia_df

print("dict_to_dataframe() defined.")


dict_to_dataframe() defined.


## Step 4 — Add ISO3 codes for party1 / party2

Look up the ISO 3166-1 alpha-3 code for each country name using the `iso_2024-04-09.xlsx` reference table, inserting `party1_iso` and `party2_iso` columns immediately after the respective name columns.


In [6]:
def add_iso_columns(df: pd.DataFrame, iso_xlsx: str) -> pd.DataFrame:
    """Look up ISO3 codes for party1 and party2."""
    iso_df           = pd.read_excel(iso_xlsx)
    iso_df.columns   = ["ISO", "country"]
    iso_df["country"] = iso_df["country"].apply(country_name_change)
    iso_map = dict(zip(iso_df["country"], iso_df["ISO"]))
    df.insert(df.columns.get_loc("party1") + 1, "party1_iso", df["party1"].map(iso_map))
    df.insert(df.columns.get_loc("party2") + 1, "party2_iso", df["party2"].map(iso_map))
    return df

print("add_iso_columns() defined.")


add_iso_columns() defined.


## Step 5 — Standardise column values

Map raw JSON label strings to UNCTAD IIABD canonical values for the 14 columns used in the `protective` indicator, and rename metadata columns to clean snake_case names.


In [7]:
VALUE_MAPS = {
    "FET_Type":            {"FET unqualified": "FET unqualified", "FET qualified": "FET qualified",
                            "No FET clause": "None", "None": "None"},
    "Exprop_Scope":        {"Indirect expropriation mentioned": "Indirect expropriation mentioned",
                            "Indirect expropriation not mentioned": "Indirect expropriation not mentioned"},
    "Exprop_IndirectDef":  {"Yes": "Yes", "No": "No"},
    "Exprop_CarveoutReg":  {"Yes": "Yes", "No": "No"},
    "Preamble_RightReg":   {"Yes": "Yes", "No": "No"},
    "Preamble_Env":        {"Yes": "Yes", "No": "No"},
    "Mention_HealthEnv":   {"Yes": "Yes", "No": "No"},
    "Mention_RightReg":    {"Yes": "Yes", "No": "No"},
    "Exception_HealthEnv": {"Yes": "Yes", "No": "No", "Not applicable": "Not applicable",
                             "Inconclusive": "Inconclusive"},
    "DoB_Clause":          {"Yes": "Yes", "No": "No", "Inconclusive": "Inconclusive"},
    "ISDS":                {"Yes": "Yes", "No": "No"},
    "ISDS_LimitProvisions":{"Yes": "Yes", "No": "No", "Not applicable": "Not applicable",
                             "Inconclusive": "Inconclusive"},
    "ISDS_LimitAreas":     {"Yes": "Yes", "No": "No", "Not applicable": "Not applicable"},
    "Term_SunsetLength":   {"10 years": "10 years", "15 years": "15 years",
                             "20 years": "20 years", "5 years": "5 years",
                             "None": "None", "Other": "Other",
                             "Not applicable": "Not applicable", "Inconclusive": "Inconclusive"},
}

META_RENAME = {
    "file_name":                "file_name",
    "Treaty type":              "treatytype",
    "Status":                   "status",
    "Date of signature":        "dateofsignature",
    "Date of entry into force": "dateofentryintoforce",
    "Treaty full text":         "treatyfulltext",
    "IIA content":              "iiacontent",
    "Facilitation content":     "facilitationcontent",
    "Originally mapped":        "originallymapped",
    "Mapped by":                "mappedby",
}


def standardise_values(df: pd.DataFrame) -> pd.DataFrame:
    """Apply VALUE_MAPS and META_RENAME; lowercase all column names."""
    df = df.rename(columns=META_RENAME)
    for col, vmap in VALUE_MAPS.items():
        if col in df.columns:
            df[col] = df[col].map(lambda v, m=vmap: m.get(str(v).strip(), str(v).strip()))
    df.columns = [c.lower() for c in df.columns]
    return df

print("standardise_values() defined.")


standardise_values() defined.


## Step 6 — `isds_limitprovisions` special-case overrides

Eight treaties have `isds='Yes'` but `isds_limitprovisions='Not applicable'` in the raw data due to a coding ambiguity. The corrected values below were verified by the research team against the treaty texts.


In [8]:
ISDS_LIMITPROV_OVERRIDES = [
    # (party_A_substring,  party_B_substring,  corrected_value)
    ("belarus",    "georgia",      "No"),
    ("belarus",    "hungary",      "No"),
    ("belarus",    "uzbekistan",   "Yes"),
    ("cabo verde", "switzerland",  "No"),
    ("algeria",    "germany",      "No"),
    ("algeria",    "romania",      "No"),
    ("hungary",    "tajikistan",   "No"),
    ("iran",       "hungary",      "No"),
]


def apply_isds_overrides(df: pd.DataFrame) -> pd.DataFrame:
    """Apply the eight verified isds_limitprovisions corrections in-place."""
    col = "isds_limitprovisions"
    for p1, p2, val in ISDS_LIMITPROV_OVERRIDES:
        mask = (
            (df["party1"].str.contains(p1, case=False, na=False) &
             df["party2"].str.contains(p2, case=False, na=False)) |
            (df["party1"].str.contains(p2, case=False, na=False) &
             df["party2"].str.contains(p1, case=False, na=False))
        )
        n = mask.sum()
        if n > 0:
            df.loc[mask, col] = val
            print(f"  Override {p1.title()}-{p2.title()} → '{val}'  ({n} row(s))")
        else:
            print(f"  WARNING: no row found for {p1}-{p2}")
    return df

print("apply_isds_overrides() defined.")


apply_isds_overrides() defined.


## Step 7 — Compute the 14-criterion `protective` indicator

A treaty is classified as `protective = 1` if and only if **all 14** of the following criteria are satisfied:

| # | Criterion | Required value |
|---|-----------|----------------|
| 1 | `fet_type` | `FET unqualified` |
| 2 | `exprop_scope` | `Indirect expropriation mentioned` |
| 3 | `exprop_indirectdef` | `No` |
| 4 | `exprop_carveoutreg` | `No` |
| 5 | `preamble_rightreg` | `No` |
| 6 | `preamble_env` | `No` |
| 7 | `mention_healthenv` | `No` |
| 8 | `mention_rightreg` | `No` |
| 9 | `exception_healthenv` | `No` or `Not applicable` |
| 10 | `dob_clause` | `No` |
| 11 | `isds` | `Yes` |
| 12 | `isds_limitprovisions` | `No` or `Not applicable` |
| 13 | `isds_limitareas` | `No` or `Not applicable` |
| 14 | `term_sunsetlength` | `10 years`, `15 years`, or `20 years` |


In [9]:
def compute_protective(df: pd.DataFrame) -> pd.DataFrame:
    """Add the binary 'protective' column implementing the 14 criteria."""
    c = pd.DataFrame(index=df.index)
    c["c1"]  = df["fet_type"].eq("FET unqualified")
    c["c2"]  = df["exprop_scope"].eq("Indirect expropriation mentioned")
    c["c3"]  = df["exprop_indirectdef"].eq("No")
    c["c4"]  = df["exprop_carveoutreg"].eq("No")
    c["c5"]  = df["preamble_rightreg"].eq("No")
    c["c6"]  = df["preamble_env"].eq("No")
    c["c7"]  = df["mention_healthenv"].eq("No")
    c["c8"]  = df["mention_rightreg"].eq("No")
    c["c9"]  = df["exception_healthenv"].isin(["No", "Not applicable"])
    c["c10"] = df["dob_clause"].eq("No")
    c["c11"] = df["isds"].eq("Yes")
    c["c12"] = df["isds_limitprovisions"].isin(["No", "Not applicable"])
    c["c13"] = df["isds_limitareas"].isin(["No", "Not applicable"])
    c["c14"] = df["term_sunsetlength"].isin(["10 years", "15 years", "20 years"])

    df["protective"] = c.all(axis=1).astype(int)

    labels = {
        "c1":  "FET unqualified",          "c2":  "Indirect exprop mentioned",
        "c3":  "Exprop not defined",        "c4":  "No exprop carve-out",
        "c5":  "No preamble R2R",           "c6":  "No preamble env",
        "c7":  "No mention health/env",     "c8":  "No mention R2R",
        "c9":  "No health/env exception",   "c10": "No DoB clause",
        "c11": "ISDS = Yes",               "c12": "No ISDS limit (provisions)",
        "c13": "No ISDS limit (areas)",    "c14": "Sunset >= 10 years",
    }
    print("  Criterion pass rates (% of all treaties):")
    for k, label in labels.items():
        print(f"    {k}  {label:<35}  {100 * c[k].mean():5.1f}%")

    n1 = df["protective"].sum()
    print(f"\n  protective = 1 : {n1} ({100*n1/len(df):.1f}%)")
    print(f"  protective = 0 : {len(df)-n1} ({100*(len(df)-n1)/len(df):.1f}%)")
    return df

print("compute_protective() defined.")


compute_protective() defined.


## Step 8 — Merge OECD bilateral FDI positions

Source: OECD SDMX REST API, dataflow `OECD.DAF.INV:DSD_FDI@DF_FDI_POS_CTRY`. Outward directional FDI positions, BMD4, all activities, immediate counterpart, annual 2021–2024, USD millions, `TYPE_ENTITY=ALL`.

For each BIT pair `(party1_iso, party2_iso)`:
- `fdi_pos_1_to_2_c` = mean annual outward FDI from party1 → party2 (USD millions)  
- `fdi_pos_2_to_1_c` = mean annual outward FDI from party2 → party1 (USD millions)


In [10]:
def merge_oecd_fdi(df: pd.DataFrame, oecd_fdi_csv: str) -> pd.DataFrame:
    """Compute mean bilateral FDI positions (2021-2024) and merge onto BIT dataset."""
    ISO_HARMONISE = {"ROM": "ROU"}   # Romania: legacy → ISO 3166-1 current

    print(f"  Loading OECD FDI data from {oecd_fdi_csv} …")
    raw = pd.read_csv(oecd_fdi_csv, low_memory=False)
    fdi = raw[raw["TYPE_ENTITY"] == "ALL"].copy()
    fdi = fdi.dropna(subset=["OBS_VALUE"])
    fdi = fdi[["REF_AREA", "COUNTERPART_AREA", "TIME_PERIOD", "OBS_VALUE"]].copy()
    fdi.columns = ["reporter_iso", "partner_iso", "year", "fdi_mUSD"]

    fdi_mean = (
        fdi.groupby(["reporter_iso", "partner_iso"])["fdi_mUSD"]
        .mean().reset_index().rename(columns={"fdi_mUSD": "fdi_mean_mUSD"})
    )
    fdi_lookup = {
        (row.reporter_iso, row.partner_iso): row.fdi_mean_mUSD
        for row in fdi_mean.itertuples(index=False)
    }

    def _lookup(iso_a, iso_b):
        a = ISO_HARMONISE.get(iso_a, iso_a)
        b = ISO_HARMONISE.get(iso_b, iso_b)
        return fdi_lookup.get((a, b), np.nan)

    df["fdi_pos_1_to_2_c"] = df.apply(lambda r: _lookup(r["party1_iso"], r["party2_iso"]), axis=1)
    df["fdi_pos_2_to_1_c"] = df.apply(lambda r: _lookup(r["party2_iso"], r["party1_iso"]), axis=1)

    n12 = df["fdi_pos_1_to_2_c"].notna().sum()
    n21 = df["fdi_pos_2_to_1_c"].notna().sum()
    print(f"  fdi_pos_1_to_2_c matched: {n12} / {len(df)} treaties")
    print(f"  fdi_pos_2_to_1_c matched: {n21} / {len(df)} treaties")
    return df

print("merge_oecd_fdi() defined.")


merge_oecd_fdi() defined.


## Step 9 — Compute `EXT_BD_c` from raw IUCN species data

Computes the range-weighted extinction-risk index for all 254 countries covered by IUCN data, then merges onto the BIT dataset by ISO3 code for both parties.


In [11]:
STATUS_WEIGHTS = {
    "LC": 0.0, "NT": 0.2, "VU": 0.4, "EN": 0.6,
    "CR": 0.8, "CR(PE)": 0.8, "CR(PEW)": 0.8,
    "EX": 1.0, "EW": 1.0,
    # DD intentionally omitted → excluded from calculation
}


def compute_ext_bd_c(spp_list_csv: str, spp_pp_csv: str) -> pd.Series:
    """Compute the EXT_BD_c index for every country from raw IUCN species data."""
    print(f"  Loading species assessments from {spp_list_csv} …")
    spp_list = pd.read_csv(spp_list_csv, usecols=["TaxonID", "Cat_latest"])
    spp_list["weight"] = spp_list["Cat_latest"].map(STATUS_WEIGHTS)
    spp_list = spp_list.dropna(subset=["weight"])
    print(f"  {len(spp_list):,} species with valid IUCN status ({spp_list['Cat_latest'].value_counts().to_dict()})")

    print(f"  Loading species-country presence data from {spp_pp_csv} …")
    spp_pp = pd.read_csv(spp_pp_csv, usecols=["TaxonID", "ISO3", "pp"])
    print(f"  {len(spp_pp):,} species-country records, "          f"{spp_pp['ISO3'].nunique()} countries, {spp_pp['TaxonID'].nunique():,} species")

    merged = spp_pp.merge(spp_list[["TaxonID", "weight"]], on="TaxonID", how="inner")
    ecost  = merged.assign(wp=merged["weight"] * merged["pp"]).groupby("ISO3")["wp"].sum()

    max_ecost = ecost.max()
    ext_bd_c  = ecost / max_ecost
    print(f"  E_cost max: {max_ecost:.4f} (country: {ecost.idxmax()})")
    print(f"  EXT_BD_c range: [{ext_bd_c.min():.6f}, {ext_bd_c.max():.6f}]")
    return ext_bd_c


def merge_ext_bd_c(df: pd.DataFrame, spp_list_csv: str, spp_pp_csv: str) -> pd.DataFrame:
    """Compute EXT_BD_c and merge onto BIT dataset for both parties."""
    ext_bd_c = compute_ext_bd_c(spp_list_csv, spp_pp_csv)
    df["ext_bd_c_party1"] = df["party1_iso"].map(ext_bd_c)
    df["ext_bd_c_party2"] = df["party2_iso"].map(ext_bd_c)
    print(f"  ext_bd_c_party1 matched: {df['ext_bd_c_party1'].notna().sum()} / {len(df)}")
    print(f"  ext_bd_c_party2 matched: {df['ext_bd_c_party2'].notna().sum()} / {len(df)}")
    return df

print("compute_ext_bd_c() and merge_ext_bd_c() defined.")


compute_ext_bd_c() and merge_ext_bd_c() defined.


## Step 10 — Compute `t_treaties` indicator

A treaty is a **T-treaty** (`t_treaties = 1`) if and only if:
- `protective = 1`, **AND**
- at least one directional FDI flow exceeds the threshold **while** the receiving country has meaningful biodiversity exposure:
  - `(fdi_pos_1_to_2_c > 1,000 USD mn  AND  ext_bd_c_party2 ≥ 0.1)`  
    **OR**  
  - `(fdi_pos_2_to_1_c > 1,000 USD mn  AND  ext_bd_c_party1 ≥ 0.1)`


In [12]:
def compute_t_treaties(
        df: pd.DataFrame,
        fdi_thr: float = FDI_THRESHOLD_mUSD,
        ext_thr: float = EXT_THRESHOLD) -> pd.DataFrame:
    """Add the binary 't_treaties' column."""
    flow_1_to_2 = df["fdi_pos_1_to_2_c"].gt(fdi_thr) & df["ext_bd_c_party2"].ge(ext_thr)
    flow_2_to_1 = df["fdi_pos_2_to_1_c"].gt(fdi_thr) & df["ext_bd_c_party1"].ge(ext_thr)
    df["t_treaties"] = (df["protective"].eq(1) & (flow_1_to_2 | flow_2_to_1)).astype(int)

    n1 = df["t_treaties"].sum()
    print(f"  t_treaties = 1 : {n1} ({100*n1/len(df):.1f}%)")
    print(f"  t_treaties = 0 : {len(df)-n1} ({100*(len(df)-n1)/len(df):.1f}%)")
    print(f"  (thresholds: FDI > {fdi_thr:,} USD mn,  EXT_BD_c >= {ext_thr})")
    return df

print("compute_t_treaties() defined.")


compute_t_treaties() defined.


## Step 11 — Country-group indicators and column renaming

Adds binary indicators for **LDC** (UN 2025 list), **megadiverse** (UNEP-WCMC 17 countries), and **OECD** membership (38 members, 2025) for both parties. Then renames all columns to < 30 characters with intuitive snake_case names.


In [13]:
LDC_ISO3 = {
    "AFG","AGO","BGD","BEN","BTN","BFA","BDI","KHM","CAF","TCD",
    "COM","COD","DJI","ERI","ETH","GMB","GIN","GNB","HTI","KIR",
    "LAO","LSO","LBR","MDG","MWI","MLI","MRT","MOZ","MMR","NPL",
    "NER","RWA","STP","SEN","SLE","SLB","SOM","SSD","SDN","TLS",
    "TGO","TUV","UGA","TZA","YEM","ZMB",
}

MEGA_ISO3 = {"AUS","BRA","CHN","COL","COD","ECU","IND","IDN",
             "MDG","MYS","MEX","PNG","PER","PHL","ZAF","USA","VEN"}

OECD_ISO3 = {
    "AUS","AUT","BEL","CAN","CHL","COL","CRI","CZE","DNK","EST",
    "FIN","FRA","DEU","GRC","HUN","ISL","IRL","ISR","ITA","JPN",
    "KOR","LVA","LTU","LUX","MEX","NLD","NZL","NOR","POL","PRT",
    "SVK","SVN","ESP","SWE","CHE","TUR","GBR","USA",
}


def add_country_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Add binary LDC, megadiverse, and OECD indicators for party1 and party2."""
    for iso_col, suffix in [("party1_iso", "party1"), ("party2_iso", "party2")]:
        iso = df[iso_col].fillna("")
        df[f"ldc_{suffix}"]    = iso.isin(LDC_ISO3).astype(int)
        df[f"megaBD_{suffix}"] = iso.isin(MEGA_ISO3).astype(int)
        df[f"oecd_{suffix}"]   = iso.isin(OECD_ISO3).astype(int)
    for grp in ("ldc", "megaBD", "oecd"):
        n1 = df[f"{grp}_party1"].sum()
        n2 = df[f"{grp}_party2"].sum()
        print(f"  {grp:8s}  party1={n1:4d}  party2={n2:4d} treaties with ≥1 member country")
    return df

print("add_country_indicators() defined.")


add_country_indicators() defined.


In [14]:
# Full column-rename mapping (abbreviated for readability — see .py source for all entries)
COLUMN_RENAME = {
    "facilitation content":      "facilitation_content",
    "about":                     "treaty_about",
    "dobdiscrmand":              "dob_discr_mand",
    "cont_inacchoststatelawsreq":"cont_host_laws_req",
    "prohib_unrarbdiscmeas":     "prohib_arb_disc_meas",
    # MFN
    ("standards of treatment - most-favoured-nation (mfn) treatment -- "
     "exceptions from mfn obligation --- procedural issues (isds) or "
     "substantive iia standards"): "mfn_excep_isds_substance",
    # Digitalisation
    ("digitalization of the investment environment - "
     "electronic publication of investment measures"): "digit_elec_publication",
    ("digitalization of the investment environment - "
     "electronic application/documents for authorizations"): "digit_elec_application",
    ("digitalization of the investment environment - "
     "electronic payment for authorizations"): "digit_elec_payment",
    ("digitalization of the investment environment - investors or key "
     "personnel entry procedure: online information or application"): "digit_entry_online_info",
    ("digitalization of the investment environment - "
     "local supplier databases/development programmes"): "digit_supplier_db",
    # Investment facilitation type
    ("type of investment facilitation in the iia - "
     "investment facilitation iia, chapters or provisions"): "invfac_type_chapters",
    # Transparency
    "transparency of investment measures - chapter or individual provisions": "trans_inv_chapters",
    "transparency of investment measures - scope of publication commitment": "trans_pub_scope",
    "transparency of investment measures - electronic publication": "trans_elec_pub",
    "transparency of investment measures - publication of proposed measures": "trans_pub_proposed",
    ("transparency of investment measures - "
     "opportunity to comment on proposed measures"): "trans_comment_proposed",
    "transparency of investment measures - inquiry mechanisms": "trans_inquiry_mechs",
    ("transparency of investment measures - "
     "exceptions from publication commitments"): "trans_pub_exceptions",
    "transparency of investment measures - subject to adr/consultations": "trans_subject_adr",
    "transparency of investment measures - subject to dispute settlement": "trans_subject_isds",
    # Administration
    ("administration of investment measures - "
     "independent competent authority"): "admin_indep_authority",
    ("administration of investment measures - "
     "objective and impartial administration"): "admin_impartial",
    "administration of investment measures - appeal and review": "admin_appeal_review",
    # Streamlining
    ("streamlining of investment procedures - "
     "chapter or individual provisions"): "stream_proc_chapters",
    "streamlining of investment procedures - type of commitment": "stream_proc_type",
    "streamlining of investment procedures - procedures covered": "stream_proc_covered",
    ("streamlining of investment procedures - "
     "avoiding multiple applications"): "stream_avoid_multi_apps",
    ("streamlining of investment procedures - "
     "reasonable processing time and fees"): "stream_proc_time_fees",
    ("streamlining of investment procedures - "
     "application processing requirements"): "stream_proc_requirements",
    ("streamlining of investment procedures - "
     "electronic application/documents"): "stream_elec_application",
    "streamlining of investment procedures - electronic payment": "stream_elec_payment",
    ("streamlining of investment procedures - "
     "subject to adr/consultations"): "stream_subject_adr",
    ("streamlining of investment procedures - "
     "subject to dispute settlement"): "stream_subject_isds",
    # Regulatory practices
    "regulatory practices - chapter or individual provisions": "regprac_chapters",
    "regulatory practices - type of commitment": "regprac_type",
    "regulatory practices - measures covered": "regprac_measures",
    "regulatory practices - domestic inter-agency coordination": "regprac_interagency",
    "regulatory practices - regulatory impact assessment": "regprac_impact_assess",
    ("regulatory practices - "
     "periodic review of regulatory measures"): "regprac_periodic_review",
    "regulatory practices - transparency of the regulatory process": "regprac_transparency",
    "regulatory practices - regulatory cooperation": "regprac_cooperation",
    "regulatory practices - subject to adr/consultations": "regprac_subject_adr",
    "regulatory practices - subject to dispute settlement": "regprac_subject_isds",
    # Entry and stay
    ("entry and temporary stay of investors and key personnel - "
     "chapter or individual provisions"): "entry_stay_chapters",
    ("entry and temporary stay of investors and key personnel - "
     "type of commitment"): "entry_stay_type",
    ("entry and temporary stay of investors and key personnel - "
     "persons covered"): "entry_stay_persons",
    ("entry and temporary stay of investors and key personnel - "
     "transparency for entry/stay procedures"): "entry_stay_transparency",
    ("entry and temporary stay of investors and key personnel - "
     "streamlining of entry/stay procedures"): "entry_stay_streamlining",
    ("entry and temporary stay of investors and key personnel - "
     "online information or application"): "entry_stay_online_info",
    ("entry and temporary stay of investors and key personnel - "
     "commitment to grant entry (liberalization)"): "entry_stay_liberalize",
    ("entry and temporary stay of investors and key personnel - "
     "subject to adr/consultations"): "entry_stay_subject_adr",
    ("entry and temporary stay of investors and key personnel - "
     "subject to dispute settlement"): "entry_stay_subject_isds",
    # Proactive facilitation
    ("proactive facilitation of sustainable investment - "
     "promotion/facilitation of sustainable investment: activities"): "profac_sustinv_activities",
    ("proactive facilitation of sustainable investment - "
     "promotion/facilitation of sustainable investment: types of investment"): "profac_sustinv_types",
    # Investor engagement
    ("investor engagement and local supplier linkages - "
     "focal point: information services"): "inveng_focal_info",
    ("investor engagement and local supplier linkages - "
     "focal point: advisory/grievance services"): "inveng_focal_advisory",
    ("investor engagement and local supplier linkages - "
     "local supplier databases/development programmes"): "inveng_supplier_db",
    # Facilitation cooperation
    ("facilitation-related cooperation and development provisions - "
     "institutional framework (investment-specific committee)"): "facdvp_inst_framework",
    ("facilitation-related cooperation and development provisions - "
     "implementation or monitoring programme"): "facdvp_monitoring",
    ("facilitation-related cooperation and development provisions - "
     "cooperation between investment agencies (e.g. ipas)"): "facdvp_agency_coop",
    ("facilitation-related cooperation and development provisions - "
     "special and differential treatment"): "facdvp_special_diff_treat",
    ("facilitation-related cooperation and development provisions - "
     "technical assistance / capacity building"): "facdvp_tech_assistance",
}


def rename_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Rename all columns to < 30 characters with intuitive names."""
    df   = df.rename(columns=COLUMN_RENAME)
    long = [c for c in df.columns if len(c) >= 30]
    if long:
        print(f"  WARNING: {len(long)} column(s) still >= 30 chars:")
        for c in long:
            print(f"    {len(c):3d}  {c}")
    else:
        print(f"  All {len(df.columns)} column names are < 30 characters.")
    return df

print("add_country_indicators(), COLUMN_RENAME, and rename_columns() defined.")


add_country_indicators(), COLUMN_RENAME, and rename_columns() defined.


## Step 12 — Run the full pipeline

Execute all steps in sequence to produce the final dataset. The cell below runs the complete pipeline and saves the output CSV.


In [15]:
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Step 1: Load raw IIA dictionary
    print("Step 1/11  Loading IIA dictionary …")
    with open(IIA_DICT_PATH, "r", encoding="utf-8") as f:
        iia_dict_raw = json.load(f)
    total_raw = len(iia_dict_raw.get("iias", {}))
    print(f"  Loaded {total_raw} IIA entries from {IIA_DICT_FILE}")

    # Step 2: Flatten
    print("\nStep 2/11  Flattening nested dictionary …")
    flat = flatten_iia_dict(iia_dict_raw)
    print(f"  Flattened {len(flat)} treaties (skipped malformed entries)")

    # Step 3: Convert to DataFrame
    print("\nStep 3/11  Building DataFrame and renaming columns …")
    df = dict_to_dataframe(flat, RENAME_CSV)
    print(f"  Shape after rename: {df.shape}")

    # Step 4: Add ISO codes
    print("\nStep 4/11  Adding ISO3 codes …")
    df = add_iso_columns(df, ISO_XLSX)
    print(f"  party1_iso missing: {df['party1_iso'].isna().sum()} | "          f"party2_iso missing: {df['party2_iso'].isna().sum()}")

    # Step 5: Standardise values
    print("\nStep 5/11  Standardising values and column names …")
    df = standardise_values(df)

    # Step 6: Filter to BITs in force with both ISO codes resolved
    print("\nStep 6/11  Filtering to BITs, In force, both parties ISO-resolved …")
    n_before = len(df)
    df = df[
        (df["treatytype"] == "Bilateral Investment Treaties") &
        (df["status"]     == "In force") &
        (df["party1_iso"].notna()) &
        (df["party2_iso"].notna())
    ].copy().reset_index(drop=True)
    print(f"  {n_before} → {len(df)} rows  (removed {n_before - len(df)})")

    # Step 7: Overrides + protective indicator
    print("\nStep 7/11  Applying ISDS overrides and computing 'protective' …")
    df = apply_isds_overrides(df)
    df = compute_protective(df)

    # Step 8: Merge OECD FDI
    print("\nStep 8/11  Merging OECD bilateral FDI positions …")
    df = merge_oecd_fdi(df, OECD_FDI_CSV)

    # Step 9: EXT_BD_c
    print("\nStep 9/11  Computing EXT_BD_c from raw IUCN species data …")
    df = merge_ext_bd_c(df, SPP_LIST_CSV, SPP_PP_CSV)

    # Step 10: T_treaties
    print("\nStep 10/11  Computing T_treaties …")
    df = compute_t_treaties(df)

    # Step 10b: Country-group indicators
    print("\nStep 10b/11  Adding country-group indicators …")
    df = add_country_indicators(df)

    # Drop unwanted columns
    df = df.drop(columns=[c for c in ["related_agreements"] if c in df.columns])

    # Step 11: Rename columns
    print("\nStep 11/11  Renaming columns …")
    df = rename_columns(df)

    # Sanitise strings (remove embedded newlines for CSV/Stata compatibility)
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(
        lambda col: col.str.replace(r"[\r\n]+", " ", regex=True)
    )

    # Save
    df = df.copy()  # de-fragment before writing (silences PerformanceWarning downstream)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✔  Saved {len(df)} rows × {len(df.columns)} columns")
    print(f"   → {OUTPUT_CSV}")
    return df


df = main()


Step 1/11  Loading IIA dictionary …
  Loaded 2808 IIA entries from iia_dict_26-02-12.json

Step 2/11  Flattening nested dictionary …
  Flattened 2808 treaties (skipped malformed entries)

Step 3/11  Building DataFrame and renaming columns …
  Shape after rename: (2808, 179)

Step 4/11  Adding ISO3 codes …


  party1_iso missing: 39 | party2_iso missing: 23

Step 5/11  Standardising values and column names …

Step 6/11  Filtering to BITs, In force, both parties ISO-resolved …
  2808 → 1837 rows  (removed 971)

Step 7/11  Applying ISDS overrides and computing 'protective' …
  Override Belarus-Georgia → 'No'  (1 row(s))
  Override Belarus-Hungary → 'No'  (1 row(s))
  Override Belarus-Uzbekistan → 'Yes'  (1 row(s))
  Override Cabo Verde-Switzerland → 'No'  (1 row(s))
  Override Algeria-Germany → 'No'  (1 row(s))
  Override Algeria-Romania → 'No'  (1 row(s))
  Override Hungary-Tajikistan → 'No'  (1 row(s))
  Override Iran-Hungary → 'No'  (1 row(s))
  Criterion pass rates (% of all treaties):
    c1  FET unqualified                       75.5%
    c2  Indirect exprop mentioned             97.6%
    c3  Exprop not defined                    93.4%
    c4  No exprop carve-out                   94.8%
    c5  No preamble R2R                       98.6%
    c6  No preamble env                       9

  fdi_pos_1_to_2_c matched: 611 / 1837 treaties
  fdi_pos_2_to_1_c matched: 594 / 1837 treaties

Step 9/11  Computing EXT_BD_c from raw IUCN species data …
  Loading species assessments from /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data/full_species_list_dec2023_final_240108_renamed.csv …
  24,245 species with valid IUCN status ({'LC': 16088, 'VU': 2361, 'EN': 2275, 'NT': 1988, 'CR': 1126, 'EX': 204, 'CR(PE)': 186, 'EW': 13, 'CR(PEW)': 4})
  Loading species-country presence data from /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data/spp_pp_2023_final_240111_renamed.csv …
  171,655 species-country records, 254 countries, 26,107 species
  E_cost max: 252.3844 (country: IDN)
  EXT_BD_c range: [0.000001, 1.000000]
  ext_bd_c_party1 matched: 1826 / 1837
  ext_bd_c_party2 matched: 1809 / 1837

Step 10/11  Computing T_treaties …
  t_treaties = 1 : 45 (2.4

## Validation

Quick checks against the reference dataset `IIABD_t_treaties_20260323.csv`, if available, and a summary of key output statistics.


In [16]:
# ── Key output statistics ─────────────────────────────────────────────────────
print(f"Total treaties      : {len(df)}")
print(f"Protective (= 1)    : {df['protective'].sum()}")
print(f"T-treaties  (= 1)   : {df['t_treaties'].sum()}")
print(f"Columns             : {len(df.columns)}")
print()

# Top-10 countries by EXT_BD_c
ext_top = (
    df[["party1", "party1_iso", "ext_bd_c_party1"]]
    .rename(columns={"party1": "country", "party1_iso": "ISO", "ext_bd_c_party1": "EXT_BD_c"})
    .dropna().drop_duplicates("ISO").nlargest(10, "EXT_BD_c").reset_index(drop=True)
)
print("Top-10 countries by EXT_BD_c:")
display(ext_top)


Total treaties      : 1837
Protective (= 1)    : 961
T-treaties  (= 1)   : 45
Columns             : 192

Top-10 countries by EXT_BD_c:


,country,ISO,EXT_BD_c
0,indonesia,IDN,1.000000
1,mexico,MEX,0.991080
2,brazil,BRA,0.961620
3,colombia,COL,0.897898
4,madagascar,MDG,0.722472
5,india,IND,0.576318
6,china,CHN,0.568209
7,australia,AUS,0.522965
8,united states,USA,0.493069
9,peru,PER,0.490947


In [17]:
# ── Optional: compare against reference dataset ───────────────────────────────
import os
ref_path = os.path.join(DATA_DIR, "IIABD_t_treaties_20260323.csv")
if os.path.exists(ref_path):
    ref = pd.read_csv(ref_path, low_memory=False)
    print("── Validation against reference dataset ──")
    print(f"  Replicated total treaties  : {len(df)}")
    print(f"  Reference  total treaties  : {len(ref)}")
    if "protective" in ref.columns:
        print(f"  Replicated protective = 1  : {df['protective'].sum()}")
        print(f"  Reference  protective = 1  : {ref['protective'].sum()}")
        print(f"  Match: {df['protective'].sum() == ref['protective'].sum()}")
    if "T_treaties" in ref.columns:
        print(f"  Replicated t_treaties = 1  : {df['t_treaties'].sum()}")
        print(f"  Reference  T_treaties = 1  : {ref['T_treaties'].sum()}")
else:
    print(f"Reference dataset not found at: {ref_path}")
    print("Skipping validation.")


Reference dataset not found at: /Users/marksan/Library/CloudStorage/OneDrive-Personal/KTH/research projects/Investment_treaties/data/replication_data/IIABD_t_treaties_20260323.csv
Skipping validation.


## Step 13 — OECD treaties: identification and report

Flags every treaty to which **at least one party is an OECD member** as `oecd_treaty`, and reports (i) the number of such OECD treaties and (ii) how many of them are *protective*. This mirrors the "OECD treaties" universe used in the paper (treaties involving ≥1 OECD country).

In [18]:
# ── OECD treaties: at least one party an OECD member ──────────────────────────
df = df.copy()                      # de-fragment after the many pipeline inserts
df["oecd_treaty"] = ((df["oecd_party1"] == 1) | (df["oecd_party2"] == 1)).astype(int)

n_total      = len(df)
n_oecd       = int(df["oecd_treaty"].sum())
oecd_df      = df[df["oecd_treaty"] == 1]
n_oecd_prot  = int((oecd_df["protective"] == 1).sum())
n_oecd_nonpr = int((oecd_df["protective"] == 0).sum())

# Distinct countries party to OECD treaties (OECD members vs non-OECD partners)
oecd_parties = set(oecd_df["party1_iso"].dropna()) | set(oecd_df["party2_iso"].dropna())
oecd_members = {c for c in oecd_parties if c in OECD_ISO3}

print("── OECD treaties (≥1 OECD member party) ───────────────────────")
print(f"Total treaties           : {n_total}")
print(f"OECD treaties            : {n_oecd}  ({n_oecd/n_total:.1%} of all treaties)")
print(f"  protective             : {n_oecd_prot}")
print(f"  non-protective         : {n_oecd_nonpr}")
print(f"Protective share (OECD)  : {n_oecd_prot/n_oecd:.1%}")
print(f"Distinct countries party : {len(oecd_parties)} "
      f"({len(oecd_members)} OECD members, {len(oecd_parties) - len(oecd_members)} non-OECD)")

# Compact summary table for the report
oecd_summary = pd.DataFrame(
    {"treaty_group": ["All treaties", "OECD treaties", " protective", " non-protective"],
     "count":        [n_total, n_oecd, n_oecd_prot, n_oecd_nonpr]}
)
display(oecd_summary)

# Optional: persist the oecd_treaty indicator alongside the dataset
# df.to_csv(OUTPUT_CSV, index=False)


── OECD treaties (≥1 OECD member party) ───────────────────────
Total treaties           : 1837
OECD treaties            : 1326  (72.2% of all treaties)
  protective             : 680
  non-protective         : 646
Protective share (OECD)  : 51.3%
Distinct countries party : 171 (36 OECD members, 135 non-OECD)


,treaty_group,count
0,All treaties,1837
1,OECD treaties,1326
2,protective,680
3,non-protective,646


## Step 14 — Merge tax revenue and GDP for both treaty parties

Adds, for **each party of every treaty**, that country's tax-revenue share of GDP, GDP, and annual tax revenue (USD millions), so FDI positions can be expressed relative to one year of national tax receipts (the FDI/TR ratio).

Following the paper's definition, annual tax revenue $=$ (tax-revenue share of GDP) $\times$ (2024 GDP in current USD):
* **share of GDP** comes from `tax_revenue_harmonised_2018_2024.csv` (preferred source per country: OECD Revenue Statistics $>$ UNU-WIDER GRD $>$ World Bank WDI; latest year $\leq 2024$), with the World Bank file's own share as a final fallback;
* **GDP** (`gdp_current_usd_2024`) comes from `tax_revenue_2024_worldbank.csv`.

The cell ends by printing host-level protective FDI/TR for the 19 exposed countries, for comparison with Table 3.

In [19]:
# ── Step 14: merge tax revenue (%) and GDP onto both treaty parties ──────────
# World Bank file  : iso3_code, tax_revenue_pct_gdp, gdp_current_usd_2024, ...
# Harmonised file  : iso3_code, tax_revenue_pct_gdp, data_year, source  (one row/country)
wb = pd.read_csv(TAX_WB_CSV).drop_duplicates("iso3_code").set_index("iso3_code")
hm = pd.read_csv(TAX_HARM_CSV).drop_duplicates("iso3_code").set_index("iso3_code")

gdp_usd = pd.to_numeric(wb["gdp_current_usd_2024"], errors="coerce")
# preferred tax share = harmonised; fall back to the World Bank share where harmonised is missing
pct = (pd.to_numeric(hm["tax_revenue_pct_gdp"], errors="coerce")
         .combine_first(pd.to_numeric(wb["tax_revenue_pct_gdp"], errors="coerce")))

# Per-country table (GDP and annual tax revenue in USD *millions*, to match FDI positions)
tax = pd.DataFrame({
    "tax_rev_pct":  pct,
    "gdp_musd":     gdp_usd / 1e6,
    "tax_rev_musd": pct / 100.0 * gdp_usd / 1e6,
})
print(f"Countries with tax share : {int(tax['tax_rev_pct'].notna().sum())}")
print(f"Countries with GDP       : {int(tax['gdp_musd'].notna().sum())}")
print(f"Countries with tax level : {int(tax['tax_rev_musd'].notna().sum())}")

# Merge onto both parties of every treaty
df = df.copy()
for p in ("party1", "party2"):
    iso = df[f"{p}_iso"]
    df[f"tax_rev_pct_{p}"]  = iso.map(tax["tax_rev_pct"])
    df[f"gdp_musd_{p}"]     = iso.map(tax["gdp_musd"])
    df[f"tax_rev_musd_{p}"] = iso.map(tax["tax_rev_musd"])

# Per-treaty FDI / tax-revenue (%): inward FDI into a party over its annual tax revenue
# fdi_pos_2_to_1_c = inward into party1 ; fdi_pos_1_to_2_c = inward into party2
df["fdi_tr_party1"] = 100.0 * df["fdi_pos_2_to_1_c"] / df["tax_rev_musd_party1"]
df["fdi_tr_party2"] = 100.0 * df["fdi_pos_1_to_2_c"] / df["tax_rev_musd_party2"]

# ── Validation: host-level protective FDI/TR for the 19 exposed countries ─────
HOSTS19 = {"CHN":"China","MEX":"Mexico","VNM":"Vietnam","PER":"Peru","PHL":"Philippines",
           "MYS":"Malaysia","COL":"Colombia","PAN":"Panama","HND":"Honduras","ZAF":"South Africa",
           "USA":"United States","LKA":"Sri Lanka","GTM":"Guatemala","IDN":"Indonesia",
           "TZA":"Tanzania","PNG":"Papua New Guinea","CMR":"Cameroon","AUS":"Australia","MDG":"Madagascar"}
a = df[["party1_iso","protective","fdi_pos_2_to_1_c","oecd_treaty","tax_rev_musd_party1"]].rename(
        columns={"party1_iso":"iso","fdi_pos_2_to_1_c":"fdi","tax_rev_musd_party1":"trev"})
b = df[["party2_iso","protective","fdi_pos_1_to_2_c","oecd_treaty","tax_rev_musd_party2"]].rename(
        columns={"party2_iso":"iso","fdi_pos_1_to_2_c":"fdi","tax_rev_musd_party2":"trev"})
lng = pd.concat([a, b]); lng = lng[(lng["oecd_treaty"] == 1) & (lng["protective"] == 1)]
rows = []
for iso, nm in HOSTS19.items():
    s = lng[lng["iso"] == iso]
    fdi = s["fdi"].sum(min_count=1)
    trev = s["trev"].dropna().iloc[0] if s["trev"].notna().any() else np.nan
    ratio = 100.0 * fdi / trev if (pd.notna(fdi) and pd.notna(trev) and trev) else np.nan
    rows.append((nm, fdi, trev, ratio))
chk = pd.DataFrame(rows, columns=["Host", "prot_FDI_musd", "tax_rev_musd", "FDI/TR (%)"])
print("\nHost-level protective FDI/TR (compare with Table 3):")
display(chk.round(1))

# Optional: persist the enriched dataset
# df.to_csv(OUTPUT_CSV, index=False)


Countries with tax share : 175
Countries with GDP       : 192
Countries with tax level : 164

Host-level protective FDI/TR (compare with Table 3):


,Host,prot_FDI_musd,tax_rev_musd,FDI/TR (%)
0,China,319756.8,2622820.4,12.2
1,Mexico,110042.2,292689.5,37.6
2,Vietnam,67061.4,56199.5,119.3
3,Peru,17187.5,41753.8,41.2
4,Philippines,15968.4,69695.0,22.9
5,Malaysia,15751.9,53829.7,29.3
6,Colombia,14450.3,76093.4,19.0
7,Panama,8252.3,5653.0,146.0
8,Honduras,2536.9,6637.9,38.2
9,South Africa,2399.7,105014.5,2.3
